# 03 — Gold Business Marts

**Purpose:** Business aggregations to Gold Delta marts.  
**Runtime:** Microsoft Fabric PySpark with attached default Lakehouse.  
**Status:** Provided as source; requires execution and verification in your Fabric tenant.

In [ ]:
from pyspark.sql import functions as F

items=spark.table('silver_order_items').alias('i')
orders=spark.table('silver_orders').alias('o')
products=spark.table('silver_products').alias('p')
customers=spark.table('silver_customers').alias('c')
completed=(items.join(orders,'order_id').join(products,'product_id')
    .filter(F.col('order_status')=='Completed'))

daily=(completed.groupBy(F.col('order_date').alias('sales_date'))
    .agg(F.round(F.sum('net_sales'),2).alias('net_revenue'),
         F.sum('quantity').alias('units_sold'),
         F.countDistinct('order_id').alias('orders')))
product_perf=(completed.groupBy('product_id','product_name','category')
    .agg(F.round(F.sum('net_sales'),2).alias('net_revenue'),
         F.sum('quantity').alias('units_sold'),
         F.countDistinct('order_id').alias('orders')))
customer_perf=(completed.groupBy('customer_id')
    .agg(F.round(F.sum('net_sales'),2).alias('lifetime_revenue'),
         F.countDistinct('order_id').alias('completed_orders'),
         F.max('order_date').alias('last_order_date')))
customer360=(customers.select('customer_id','customer_name','region')
    .join(customer_perf,'customer_id','left')
    .fillna({'lifetime_revenue':0.0,'completed_orders':0}))
for name,df in {'gold_daily_sales':daily,
                'gold_product_performance':product_perf,
                'gold_customer_360':customer360}.items():
    (df.write.format('delta').mode('overwrite').option('overwriteSchema','true')
       .saveAsTable(name))
    print(f'{name}: {df.count():,} rows')
print('Gold revenue:', daily.agg(F.sum('net_revenue')).first()[0])
